### Conversational Chatbot using History-Aware-Retriever

In [ ]:
import bs4
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

from langchain_community.document_loaders import WebBaseLoader
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_history_aware_retriever

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [15]:
# Loading the environment variables
load_dotenv()

# Initializing the LLM and Embedding Model
model = ChatGroq(model="llama-3.1-8b-instant")
embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

c:\Users\viren\anaconda3\envs\langchain_venv\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\viren\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 9738.34it/s]


In [19]:
# Scaping the content from a target website
loader = WebBaseLoader(
    web_paths = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
            )
        ),
    )

documents = loader.load()

In [20]:
# Splitting the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_documents = text_splitter.split_documents(documents)

In [21]:
# Embedding the text into vectors and storing it in the vectorstore
vectorstore = Chroma.from_documents(splitted_documents, embeddings)
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000017DCD06B430>, search_kwargs={})

In [22]:
## Creating the Prompt Template
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [23]:
# Creating the document and retrieval chain
document_chain = create_stuff_documents_chain(model,prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [26]:
# Invoking the retireval chain for response
response = retrieval_chain.invoke({"input":"What is Self-Reflection"})
response

{'input': 'What is Self-Reflection',
 'context': [Document(id='efc0518d-30af-40ab-b65f-9a96aece463c', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='You should only respond in JSON format as described below\nResponse Format:\n{\n    "thoughts": {\n        "text": "thought",\n        "reasoning": "reasoning",\n        "plan": "- short bulleted\\n- list that conveys\\n- long-term plan",\n        "criticism": "constructive self-criticism",\n        "speak": "thoughts summary to say to user"\n    },\n    "command": {\n        "name": "command name",\n        "args": {\n            "arg name": "value"\n        }\n    }\n}\nEnsure the response can be parsed by Python json.loads\nGPT-Engineer is another project to create a whole repository of code given a task specified in natural language. The GPT-Engineer is instructed to think over a list of smaller components to build and ask for user input to clarify questions as needed.\nHere are a sample conv

In [27]:
# Response answer
response['answer']

'{\n  "thoughts": {\n    "text": "Self-Reflection is the process of introspection and evaluation of one\'s own thoughts, feelings, and actions.",\n    "reasoning": "It involves analyzing past experiences, behaviors, and decisions to identify areas of improvement and growth.",\n    "plan": "- Identify past experiences and behaviors\\n- Evaluate their impact on current situation\\n- Determine areas for improvement"\n  },\n  "command": {\n    "name": "Define Self-Reflection"\n  }\n}'

In [29]:
# Re-invoking to check if it remembers the previous context
retrieval_chain.invoke({"input":"How do we achieve it"})['answer']

'{\n  "thoughts": {\n    "text": "We can achieve the task of planning and reacting by translating reflections and environment information into actions through the generative agent architecture.",\n    "reasoning": "Planning is used to optimize believability at the moment vs in time, considering relationships between agents and observations of one agent by another.",\n    "plan": "- Utilize the generative agent architecture to process environment information and reflections\\n- Translate processed information into actionable steps\\n- React to changing circumstances and adapt the plan accordingly",\n    "criticism": "We should continually evaluate and refine our plan to ensure it remains effective in achieving our goals.",\n    "speak": "To achieve this, we\'ll be using a generative agent architecture to process our environment and reflections, then translating that information into actionable steps."\n  },\n  "command": {\n    "name": "plan_and_react",\n    "args": {\n      "environmen

#### Adding Chat History

In [30]:
# Creating Prompt Template for making rag-chain context aware 
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [31]:
# Creating the history-aware-retriever
history_aware_retriever = create_history_aware_retriever(model, retriever, contextualize_q_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000017DCD06B430>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessag

In [32]:
# Creating the Prompt template specific for Q&A
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [33]:
# Creating the q&a chain with the history-aware-retriever
question_answer_chain = create_stuff_documents_chain(model, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [34]:
# Creating chat history record for keeping chat history
chat_history = []

# Invoking the rag chain with questions and chat history for response
question_1 = "What is Self-Reflection"
response_1 = rag_chain.invoke({"input":question_1, "chat_history":chat_history})

# Adding the chat history to the chat history record
chat_history.extend(
    [
        HumanMessage(content=question_1),
        AIMessage(content=response_1["answer"])
    ]
)

# Re-Invoking to check if the model remembers the previous conversation
question_2 = "Tell me more about it?"
response_2 = rag_chain.invoke({"input":question_2,"chat_history":chat_history})
print(response_2['answer'])

Self-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable. Self-reflection involves examining one's own thoughts, feelings, and behaviors to gain insight and understanding.


In [35]:
# Chat History Records
chat_history

[HumanMessage(content='What is Self-Reflection', additional_kwargs={}, response_metadata={}),
 AIMessage(content='{\n  "thoughts": {\n    "text": "Self-reflection is the process of examining one\'s own thoughts, feelings, and behaviors in order to gain insight and understanding.",\n    "reasoning": "Self-reflection is a key component of personal growth and development.",\n    "plan": "- Reflect on past experiences and actions\\n- Identify strengths and weaknesses\\n- Adjust plans and strategies accordingly",\n    "criticism": "Lack of self-reflection can lead to stagnation and missed opportunities.",\n    "speak": "Self-reflection is an essential tool for personal growth and self-awareness."\n  },\n  "command": {\n    "name": "define",\n    "args": {\n      "term": "Self-Reflection"\n    }\n  }\n}', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [36]:
# Creating a store variable for keeping track of session IDs & their Chat History
store = {}

# Session ID Verifier fn
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

c:\Users\viren\anaconda3\envs\langchain_venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [37]:
config = {"configurable": {"session_id": "abc123"}}  # constructs a key "abc123" in `store`

conversational_rag_chain.invoke(
    {"input": "What is Task Decomposition?"},
    config=config
)["answer"]

'Task decomposition is the process of breaking down complex tasks into smaller, more manageable subtasks or steps. It can be done using various methods, including using Language Model (LLM) with simple prompting, task-specific instructions, human inputs, or a combination of these approaches.'

In [38]:
conversational_rag_chain.invoke(
    {"input": "What are common ways of doing it?"},
    config=config,
)["answer"]

'Common ways of task decomposition include using Language Model (LLM) with simple prompting, task-specific instructions, human inputs, and the LLM+P approach which involves using an external classical planner to do long-horizon planning with the Planning Domain Definition Language (PDDL) as an intermediate interface.'

In [40]:
store['abc123']

InMemoryChatMessageHistory(messages=[HumanMessage(content='What is Task Decomposition?', additional_kwargs={}, response_metadata={}), AIMessage(content='Task decomposition is the process of breaking down complex tasks into smaller, more manageable subtasks or steps. It can be done using various methods, including using Language Model (LLM) with simple prompting, task-specific instructions, human inputs, or a combination of these approaches.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What are common ways of doing it?', additional_kwargs={}, response_metadata={}), AIMessage(content='Common ways of task decomposition include using Language Model (LLM) with simple prompting, task-specific instructions, human inputs, and the LLM+P approach which involves using an external classical planner to do long-horizon planning with the Planning Domain Definition Language (PDDL) as an intermediate interface.', additional_kwargs={}, respon